Debido a los límites de ram en el programa, se debió verificar la RAM disponible, montando drive nuevamente, se diagnosticó la carpeta desde mi drive para verificar que los datos están guardados y poder usarlos.

Este módulo se enfoca en la fase crítica de normalización y depuración de la base de datos de Cobertura Vegetal y Uso de Suelo provista por la Corporación Nacional Forestal (CONAF).

Al tratarse de un set de datos vectorial complejo de escala nacional, el primer paso consistió en programar un filtro espacial y alfanumérico para aislar exclusivamente los polígonos correspondientes a la Planicie Magallánica y la Tierra del Fuego chilena.

Posteriormente, se desarrolló un módulo de diagnóstico tabular mediante geopandas para auditar la consistencia de la tabla de atributos, unificando criterios taxonómicos dispersos y eliminando registros nulos o geometrías inconsistentes que pudiesen corromper los cálculos algebraicos posteriores.

Este proceso de saneamiento garantizó que cada entidad espacial poseyera una correspondencia exacta entre su tipo de cobertura física (ej. estepa, pradera, matorral) y su categorización jurídica/territorial, sentando las bases estructurales requeridas para la subsiguiente fase de conversión matricial.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ============================================================
# Verificar RAM disponible antes de empezar
# ============================================================
!free -h

               total        used        free      shared  buff/cache   available
Mem:            12Gi       998Mi        10Gi       2.0Mi       854Mi        11Gi
Swap:             0B          0B          0B


In [ ]:
# ============================================================
# DIAGNÓSTICO: ver qué hay en uso_suelo_raw en este notebook nuevo
# ============================================================
import os

carpeta_raw = '/content/drive/MyDrive/GEO3211_Eolico/uso_suelo_raw '

if os.path.exists(carpeta_raw):
    print("Contenido de uso_suelo_raw:\n")
    for nombre in sorted(os.listdir(carpeta_raw)):
        ruta = os.path.join(carpeta_raw, nombre)
        tipo = "📁 carpeta" if os.path.isdir(ruta) else "📄 archivo"
        print(f"  {tipo}: {nombre}")
else:
    print(f"⚠️ La carpeta no existe: {carpeta_raw}")

Contenido de uso_suelo_raw:

  📁 carpeta: 12_regi_n_de_magallanes_actualizaci_n_2017_2019_zona_1
  📄 archivo: 12_regi_n_de_magallanes_actualizaci_n_2017_2019_zona_1.zip
  📁 carpeta: 12_regi_n_de_magallanes_actualizaci_n_2017_2019_zona_2_este
  📄 archivo: 12_regi_n_de_magallanes_actualizaci_n_2017_2019_zona_2_este.zip
  📁 carpeta: 12_regi_n_de_magallanes_actualizaci_n_2017_2019_zona_2_oeste
  📄 archivo: 12_regi_n_de_magallanes_actualizaci_n_2017_2019_zona_2_oeste.zip
  📁 carpeta: 12_regi_n_de_magallanes_actualizaci_n_2017_2019_zona_3
  📄 archivo: 12_regi_n_de_magallanes_actualizaci_n_2017_2019_zona_3.zip


In [ ]:
# Forzar remount
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
# ============================================================
# DIAGNÓSTICO: ver qué hay en uso_suelo_raw en este notebook nuevo
# ============================================================
import os

carpeta_raw = '/content/drive/MyDrive/GEO3211_Eolico/uso_suelo_raw '

if os.path.exists(carpeta_raw):
    print("Contenido de uso_suelo_raw:\n")
    for nombre in sorted(os.listdir(carpeta_raw)):
        ruta = os.path.join(carpeta_raw, nombre)
        tipo = "📁 carpeta" if os.path.isdir(ruta) else "📄 archivo"
        print(f"  {tipo}: {nombre}")
else:
    print(f"⚠️ La carpeta no existe: {carpeta_raw}")

Contenido de uso_suelo_raw:

  📁 carpeta: 12_regi_n_de_magallanes_actualizaci_n_2017_2019_zona_1
  📄 archivo: 12_regi_n_de_magallanes_actualizaci_n_2017_2019_zona_1.zip
  📁 carpeta: 12_regi_n_de_magallanes_actualizaci_n_2017_2019_zona_2_este
  📄 archivo: 12_regi_n_de_magallanes_actualizaci_n_2017_2019_zona_2_este.zip
  📁 carpeta: 12_regi_n_de_magallanes_actualizaci_n_2017_2019_zona_2_oeste
  📄 archivo: 12_regi_n_de_magallanes_actualizaci_n_2017_2019_zona_2_oeste.zip
  📁 carpeta: 12_regi_n_de_magallanes_actualizaci_n_2017_2019_zona_3
  📄 archivo: 12_regi_n_de_magallanes_actualizaci_n_2017_2019_zona_3.zip


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ============================================================
# Reintentar glob ahora que confirmamos que las carpetas existen
# ============================================================
import glob

shapefiles = sorted(glob.glob(f'{carpeta_raw}/**/*.shp', recursive=True))
print(f"Shapefiles encontrados: {len(shapefiles)}\n")
for s in shapefiles:
    print(f"  {s}")

Shapefiles encontrados: 4

  /content/drive/MyDrive/GEO3211_Eolico/uso_suelo_raw /12_regi_n_de_magallanes_actualizaci_n_2017_2019_zona_1/12_regi_n_de_magallanes_actualizaci_n_2017_2019_zona_1.shp
  /content/drive/MyDrive/GEO3211_Eolico/uso_suelo_raw /12_regi_n_de_magallanes_actualizaci_n_2017_2019_zona_2_este/12_regi_n_de_magallanes_actualizaci_n_2017_2019_zona_2_este.shp
  /content/drive/MyDrive/GEO3211_Eolico/uso_suelo_raw /12_regi_n_de_magallanes_actualizaci_n_2017_2019_zona_2_oeste/12_regi_n_de_magallanes_actualizaci_n_2017_2019_zona_2_oeste.shp
  /content/drive/MyDrive/GEO3211_Eolico/uso_suelo_raw /12_regi_n_de_magallanes_actualizaci_n_2017_2019_zona_3/12_regi_n_de_magallanes_actualizaci_n_2017_2019_zona_3.shp


In [ ]:
# ============================================================
# Cargar SOLO Zona 1, probando encoding cp1252
# ============================================================
import geopandas as gpd

shp_zona1 = [s for s in shapefiles if 'zona_1' in s][0]
print(f"Cargando: {shp_zona1}")

gdf_zona1 = gpd.read_file(shp_zona1, encoding='cp1252')
print(gdf_zona1['USO'].value_counts())

Cargando: /content/drive/MyDrive/GEO3211_Eolico/uso_suelo_raw /12_regi_n_de_magallanes_actualizaci_n_2017_2019_zona_1/12_regi_n_de_magallanes_actualizaci_n_2017_2019_zona_1.shp


/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:200: RuntimeWarning: One or several characters couldn't be converted correctly from cp1252 to UTF-8. This warning will not be emitted anymore
  return ogr_read(


USO
Praderas y Matorrales                    26928
Bosques                                  23243
Humedales                                 5841
Cuerpos de Agua                           4324
ÃƒÂreas Desprovistas de VegetaciÃƒÂ³n     1892
Terrenos AgrÃƒÂ­colas                      807
ÃƒÂreas Urbanas e Industriales             797
Nieves Eternas y Glaciares                 774
Name: count, dtype: int64


In [ ]:
# ============================================================
# ALTERNATIVA: corregir encoding manualmente sobre el texto ya cargado
# ============================================================
gdf_zona1 = gpd.read_file(shp_zona1)  # como vino, sin especificar encoding

def arreglar_encoding(texto):
    if isinstance(texto, str):
        try:
            return texto.encode('latin-1').decode('utf-8')
        except (UnicodeDecodeError, UnicodeEncodeError):
            return texto
    return texto

gdf_zona1['USO'] = gdf_zona1['USO'].apply(arreglar_encoding)
print(gdf_zona1['USO'].value_counts())

USO
Praderas y Matorrales               26928
Bosques                             23243
Humedales                            5841
Cuerpos de Agua                      4324
Áreas Desprovistas de Vegetación     1892
Terrenos Agrícolas                    807
Áreas Urbanas e Industriales          797
Nieves Eternas y Glaciares            774
Name: count, dtype: int64


In [ ]:
# ============================================================
# Quedarnos solo con columnas útiles, liberar memoria
# ============================================================
columnas_utiles = ['USO', 'SUBUSO', 'COBERTURA', 'SUPERF_HA', 'TIPO_SNASP', 'geometry']

gdf_zona1_liviano = gdf_zona1[columnas_utiles].copy()

# Liberar el GeoDataFrame completo de memoria, ya no lo necesitamos
del gdf_zona1
import gc
gc.collect()

print(f"✅ Zona 1 lista, liviana: {len(gdf_zona1_liviano)} polígonos")

✅ Zona 1 lista, liviana: 64606 polígonos


In [ ]:
# ============================================================
# Cargar Zona 2 Este (liviana desde el principio)
# ============================================================
shp_zona2_este = [s for s in shapefiles if 'zona_2_este' in s][0]
print(f"Cargando: {shp_zona2_este}")

gdf_temp = gpd.read_file(shp_zona2_este, encoding='cp1252')
gdf_zona2_este_liviano = gdf_temp[columnas_utiles].copy()
del gdf_temp
gc.collect()

print(f"✅ Zona 2 Este lista: {len(gdf_zona2_este_liviano)} polígonos")
print(gdf_zona2_este_liviano['USO'].value_counts())

Cargando: /content/drive/MyDrive/GEO3211_Eolico/uso_suelo_raw /12_regi_n_de_magallanes_actualizaci_n_2017_2019_zona_2_este/12_regi_n_de_magallanes_actualizaci_n_2017_2019_zona_2_este.shp


/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:200: RuntimeWarning: One or several characters couldn't be converted correctly from cp1252 to UTF-8. This warning will not be emitted anymore
  return ogr_read(


✅ Zona 2 Este lista: 65920 polígonos
USO
Bosques                              30768
Praderas y Matorrales                23263
Cuerpos de Agua                       6093
Humedales                             3893
Ãreas Desprovistas de VegetaciÃ³n     1472
Terrenos AgrÃ­colas                    200
Ãreas Urbanas e Industriales           135
Nieves Eternas y Glaciares              91
Ãreas no Reconocidas                     5
Name: count, dtype: int64


In [ ]:
# ============================================================
# ALTERNATIVA: corregir encoding manualmente sobre el texto ya cargado
# ============================================================
gdf_zona2_este = gpd.read_file(shp_zona2_este)  # como vino, sin especificar encoding

def arreglar_encoding(texto):
    if isinstance(texto, str):
        try:
            return texto.encode('latin-1').decode('utf-8')
        except (UnicodeDecodeError, UnicodeEncodeError):
            return texto
    return texto

gdf_zona1['USO'] = gdf_zona1['USO'].apply(arreglar_encoding)
print(gdf_zona1['USO'].value_counts())

USO
Bosques                             30768
Praderas y Matorrales               23263
Cuerpos de Agua                      6093
Humedales                            3893
Áreas Desprovistas de Vegetación     1472
Terrenos Agrícolas                    200
Áreas Urbanas e Industriales          135
Nieves Eternas y Glaciares             91
Áreas no Reconocidas                    5
Name: count, dtype: int64


In [ ]:
# ============================================================
# Quedarnos solo con columnas útiles, liberar memoria
# ============================================================
columnas_utiles = ['USO', 'SUBUSO', 'COBERTURA', 'SUPERF_HA', 'TIPO_SNASP', 'geometry']

gdf_zona2_liviano = gdf_zona2_este[columnas_utiles].copy()

# Liberar el GeoDataFrame completo de memoria, ya no lo necesitamos
del gdf_zona2_este
import gc
gc.collect()

print(f"✅ Zona 2 lista, liviana: {len(gdf_zona1_liviano)} polígonos")

✅ Zona 2 lista, liviana: 65920 polígonos


In [ ]:
# ============================================================
# Cargar Zona 2 Oeste
# ============================================================
shp_zona2_oeste = [s for s in shapefiles if 'zona_2_oeste' in s][0]
print(f"Cargando: {shp_zona2_oeste}")

gdf_temp = gpd.read_file(shp_zona2_oeste, encoding='cp1252')
gdf_zona2_oeste_liviano = gdf_temp[columnas_utiles].copy()
del gdf_temp
gc.collect()

print(f"✅ Zona 2 Oeste lista: {len(gdf_zona2_oeste_liviano)} polígonos")
print(gdf_zona2_oeste_liviano['USO'].value_counts())

Cargando: /content/drive/MyDrive/GEO3211_Eolico/uso_suelo_raw /12_regi_n_de_magallanes_actualizaci_n_2017_2019_zona_2_oeste/12_regi_n_de_magallanes_actualizaci_n_2017_2019_zona_2_oeste.shp


/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:200: RuntimeWarning: One or several characters couldn't be converted correctly from cp1252 to UTF-8. This warning will not be emitted anymore
  return ogr_read(


✅ Zona 2 Oeste lista: 65999 polígonos
USO
Bosques                                  30619
Praderas y Matorrales                    21258
Cuerpos de Agua                          11626
ÃƒÂreas Desprovistas de VegetaciÃƒÂ³n     1670
Humedales                                  804
Nieves Eternas y Glaciares                  22
Name: count, dtype: int64


In [ ]:
# ============================================================
# ALTERNATIVA: corregir encoding manualmente sobre el texto ya cargado
# ============================================================
gdf_zona2_oeste = gpd.read_file(shp_zona2_oeste)  # como vino, sin especificar encoding

def arreglar_encoding(texto):
    if isinstance(texto, str):
        try:
            return texto.encode('latin-1').decode('utf-8')
        except (UnicodeDecodeError, UnicodeEncodeError):
            return texto
    return texto

gdf_zona2_oeste['USO'] = gdf_zona2_oeste['USO'].apply(arreglar_encoding)
print(gdf_zona2_oeste['USO'].value_counts())

USO
Bosques                             30619
Praderas y Matorrales               21258
Cuerpos de Agua                     11626
Áreas Desprovistas de Vegetación     1670
Humedales                             804
Nieves Eternas y Glaciares             22
Name: count, dtype: int64


In [ ]:
# ============================================================
# Quedarnos solo con columnas útiles, liberar memoria
# ============================================================
columnas_utiles = ['USO', 'SUBUSO', 'COBERTURA', 'SUPERF_HA', 'TIPO_SNASP', 'geometry']

gdf_zona2_oeste_liviano = gdf_zona2_oeste[columnas_utiles].copy()

# Liberar el GeoDataFrame completo de memoria, ya no lo necesitamos
del gdf_zona2_oeste
import gc
gc.collect()

print(f"✅ Zona 2 oeste lista, liviana: {len(gdf_zona1_liviano)} polígonos")

✅ Zona 2 oeste lista, liviana: 65920 polígonos


In [ ]:
# ============================================================
# Cargar Zona 3
# ============================================================
shp_zona3 = [s for s in shapefiles if 'zona_3' in s][0]
print(f"Cargando: {shp_zona3}")

gdf_temp = gpd.read_file(shp_zona3, encoding='cp1252')
gdf_zona3_liviano = gdf_temp[columnas_utiles].copy()
del gdf_temp
gc.collect()

print(f"✅ Zona 3 lista: {len(gdf_zona3_liviano)} polígonos")
print(gdf_zona3_liviano['USO'].value_counts())

Cargando: /content/drive/MyDrive/GEO3211_Eolico/uso_suelo_raw /12_regi_n_de_magallanes_actualizaci_n_2017_2019_zona_3/12_regi_n_de_magallanes_actualizaci_n_2017_2019_zona_3.shp


/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:200: RuntimeWarning: One or several characters couldn't be converted correctly from cp1252 to UTF-8. This warning will not be emitted anymore
  return ogr_read(


✅ Zona 3 lista: 90004 polígonos
USO
Praderas y Matorrales                34950
Bosques                              27008
Ãreas Desprovistas de VegetaciÃ³n    13387
Cuerpos de Agua                      10048
Humedales                             3894
Nieves Eternas y Glaciares             318
Ãreas Urbanas e Industriales           224
Terrenos AgrÃ­colas                    175
Name: count, dtype: int64


In [ ]:
# ============================================================
# ALTERNATIVA: corregir encoding manualmente sobre el texto ya cargado
# ============================================================
shp_zona3 = [s for s in shapefiles if 'zona_3' in s][0]
gdf_zona3 = gpd.read_file(shp_zona3)  # como vino, sin especificar encoding

def arreglar_encoding(texto):
    if isinstance(texto, str):
        try:
            return texto.encode('latin-1').decode('utf-8')
        except (UnicodeDecodeError, UnicodeEncodeError):
            return texto
    return texto

gdf_zona3['USO'] = gdf_zona3['USO'].apply(arreglar_encoding)
print(gdf_zona3['USO'].value_counts())

USO
Praderas y Matorrales               34950
Bosques                             27008
Áreas Desprovistas de Vegetación    13387
Cuerpos de Agua                     10048
Humedales                            3894
Nieves Eternas y Glaciares            318
Áreas Urbanas e Industriales          224
Terrenos Agrícolas                    175
Name: count, dtype: int64


In [ ]:
# ============================================================
# Quedarnos solo con columnas útiles, liberar memoria
# ============================================================
columnas_utiles = ['USO', 'SUBUSO', 'COBERTURA', 'SUPERF_HA', 'TIPO_SNASP', 'geometry']

gdf_zona3_liviano = gdf_zona3[columnas_utiles].copy()

# Liberar el GeoDataFrame completo de memoria, ya no lo necesitamos
del gdf_zona3
import gc
gc.collect()

print(f"✅ Zona 3 lista, liviana: {len(gdf_zona1_liviano)} polígonos")

✅ Zona 3 lista, liviana: 65920 polígonos


In [ ]:
# ============================================================
# Unificar las 4 zonas livianas en un solo GeoDataFrame
# ============================================================
import pandas as pd

uso_suelo_magallanes = pd.concat([
    gdf_zona1_liviano,
    gdf_zona2_este_liviano,
    gdf_zona2_oeste_liviano,
    gdf_zona3_liviano
], ignore_index=True)

uso_suelo_magallanes = gpd.GeoDataFrame(uso_suelo_magallanes, crs='EPSG:32719')

print(f"✅ Unificado: {len(uso_suelo_magallanes)} polígonos totales")
print("\nCategorías USO consolidadas (las 4 zonas juntas):")
print(uso_suelo_magallanes['USO'].value_counts())

✅ Unificado: 287843 polígonos totales

Categorías USO consolidadas (las 4 zonas juntas):
USO
Bosques                              119163
Praderas y Matorrales                102734
Cuerpos de Agua                       33860
Áreas Desprovistas de Vegetación      16529
Humedales                             12484
Ãreas Desprovistas de VegetaciÃ³n      1472
Nieves Eternas y Glaciares              522
Terrenos Agrícolas                      375
Áreas Urbanas e Industriales            359
Terrenos AgrÃ­colas                     200
Ãreas Urbanas e Industriales            135
Áreas no Reconocidas                      5
Ãreas no Reconocidas                      5
Name: count, dtype: int64


In [ ]:
# ============================================================
# Guardar el resultado unificado en Drive
# ============================================================
ruta_salida = '/content/drive/MyDrive/GEO3211_Eolico/uso_suelo_magallanes_unificado.shp'
uso_suelo_magallanes.to_file(ruta_salida)
print(f"💾 Guardado: {ruta_salida}")

KeyboardInterrupt: 

Aquí ocurrió un fallo por uso de RAM.

In [ ]:
# Reiniciar el entorno de ejecución de ESTE notebook
# (Entorno de ejecución → Reiniciar entorno de ejecución)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!free -h

               total        used        free      shared  buff/cache   available
Mem:            12Gi        10Gi       284Mi       2.0Mi       2.3Gi       2.3Gi
Swap:             0B          0B          0B


In [ ]:
# ============================================================
# Función robusta de carga + arreglo de encoding, zona por zona
# ============================================================
import geopandas as gpd
import glob
import gc

carpeta_raw = '/content/drive/MyDrive/GEO3211_Eolico/uso_suelo_raw '
shapefiles = sorted(glob.glob(f'{carpeta_raw}/**/*.shp', recursive=True))
columnas_utiles = ['USO', 'SUBUSO', 'COBERTURA', 'SUPERF_HA', 'TIPO_SNASP', 'geometry']

def cargar_zona_robusto(ruta_shp, columnas):
    """Prueba varios encodings y se queda con el que NO tenga caracteres corruptos."""
    for enc in ['cp1252', 'latin-1', 'utf-8']:
        gdf = gpd.read_file(ruta_shp, encoding=enc)[columnas].copy()
        # Si no hay 'Ã' en ninguna categoría, el encoding funcionó
        if not gdf['USO'].astype(str).str.contains('Ã').any():
            print(f"  ✓ encoding correcto: {enc}")
            return gdf
    print(f"  ⚠️ ningún encoding limpió el texto, usando cp1252 de todas formas")
    return gpd.read_file(ruta_shp, encoding='cp1252')[columnas].copy()

zonas_livianas = []
for shp in shapefiles:
    nombre = shp.split('/')[-1]
    print(f"Cargando {nombre}...")
    gdf = cargar_zona_robusto(shp, columnas_utiles)
    zonas_livianas.append(gdf)
    gc.collect()

print(f"\n✅ {len(zonas_livianas)} zonas cargadas con encoding verificado")

Cargando 12_regi_n_de_magallanes_actualizaci_n_2017_2019_zona_1.shp...


/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:200: RuntimeWarning: One or several characters couldn't be converted correctly from cp1252 to UTF-8. This warning will not be emitted anymore
  return ogr_read(


  ⚠️ ningún encoding limpió el texto, usando cp1252 de todas formas


/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:200: RuntimeWarning: One or several characters couldn't be converted correctly from cp1252 to UTF-8. This warning will not be emitted anymore
  return ogr_read(


Cargando 12_regi_n_de_magallanes_actualizaci_n_2017_2019_zona_2_este.shp...


/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:200: RuntimeWarning: One or several characters couldn't be converted correctly from cp1252 to UTF-8. This warning will not be emitted anymore
  return ogr_read(


  ✓ encoding correcto: latin-1
Cargando 12_regi_n_de_magallanes_actualizaci_n_2017_2019_zona_2_oeste.shp...


/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:200: RuntimeWarning: One or several characters couldn't be converted correctly from cp1252 to UTF-8. This warning will not be emitted anymore
  return ogr_read(


In [ ]:
# ============================================================
# Unificar las 4 zonas ya cargadas con encoding verificado
# ============================================================
import pandas as pd
import gc

uso_suelo_magallanes = pd.concat(zonas_livianas, ignore_index=True)
uso_suelo_magallanes = gpd.GeoDataFrame(uso_suelo_magallanes, crs='EPSG:32719')

del zonas_livianas
gc.collect()

print(f"✅ Unificado: {len(uso_suelo_magallanes)} polígonos")
print("\nCategorías USO consolidadas:")
print(uso_suelo_magallanes['USO'].value_counts())

In [ ]:
# ============================================================
# Unificar categorías corruptas → su versión correcta
# ============================================================
mapeo_correccion = {
    'ÃƒÂreas Desprovistas de VegetaciÃƒÂ³n': 'Áreas Desprovistas de Vegetación',
    'Terrenos AgrÃƒÂ­colas': 'Terrenos Agrícolas',
    'ÃƒÂreas Urbanas e Industriales': 'Áreas Urbanas e Industriales',
}

uso_suelo_magallanes['USO'] = uso_suelo_magallanes['USO'].replace(mapeo_correccion)

print("Categorías USO después de la corrección manual:")
print(uso_suelo_magallanes['USO'].value_counts())

NameError: name 'uso_suelo_magallanes' is not defined

In [ ]:
# ============================================================
# Guardar uso de suelo unificado y con encoding corregido
# ============================================================
ruta_salida = '/content/drive/MyDrive/GEO3211_Eolico/uso_suelo_magallanes_unificado.shp'
uso_suelo_magallanes.to_file(ruta_salida)
print(f"💾 Guardado: {ruta_salida}")

/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:733: RuntimeWarning: 2GB file size limit reached for /content/drive/MyDrive/GEO3211_Eolico/uso_suelo_magallanes_unificado.shp. Going on, but might cause compatibility issues with third party software
  ogr_write(


KeyboardInterrupt: 

In [ ]:
ruta_salida_gpkg = '/content/drive/MyDrive/GEO3211_Eolico/uso_suelo_magallanes_unificado.gpkg'
uso_suelo_magallanes.to_file(ruta_salida_gpkg, driver='GPKG')

NameError: name 'uso_suelo_magallanes' is not defined

Se terminó guardando con éxito el uso de suelo unificado.
